# CLASSIFICATION MODEL

This notebook builds classification models to predict whether a campaign day belongs to the **Control** or **Test** group based on marketing performance metrics.

This stage includes:

* Data Preparation & Feature Engineering
* Logistic Regression
* Decision Tree Classifier
* Random Forest Classifier
* Model Evaluation (Accuracy, Confusion Matrix, ROC-AUC)
* Feature Importance


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/processed/cleaned_marketing.csv')
df['date'] = pd.to_datetime(df['date'])

# Drop rows with missing values
df_clean = df.dropna().copy()
print(f'Dataset shape: {df_clean.shape}')
df_clean.head()

## 1. Data Preparation


In [ ]:
feature_cols = ['spend_usd', 'impressions', 'reach', 'website_clicks',
                'searches', 'view_content', 'add_to_cart', 'purchase']

X = df_clean[feature_cols]
y = (df_clean['group'] == 'test').astype(int)  # 1 = test, 0 = control

print('Class distribution:')
print(y.value_counts().rename({0: 'Control', 1: 'Test'}))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'\nTraining set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

## 2. Logistic Regression


In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print('=== Logistic Regression ===')
print(f'Accuracy: {acc_lr:.4f}')
print(f'ROC-AUC:  {auc_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Control', 'Test']))

In [ ]:
# Logistic Regression Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['Control', 'Test'],
    cmap='Blues',
    ax=ax
)
ax.set_title('Logistic Regression - Confusion Matrix')
plt.tight_layout()
plt.show()

## 3. Decision Tree Classifier


In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

acc_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)

print('=== Decision Tree ===')
print(f'Accuracy: {acc_dt:.4f}')
print(f'ROC-AUC:  {auc_dt:.4f}')
print()
print(classification_report(y_test, y_pred_dt, target_names=['Control', 'Test']))

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=feature_cols,
    class_names=['Control', 'Test'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Decision Tree (max_depth=4)')
plt.tight_layout()
plt.show()

## 4. Random Forest Classifier


In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print('=== Random Forest ===')
print(f'Accuracy: {acc_rf:.4f}')
print(f'ROC-AUC:  {auc_rf:.4f}')
print()
print(classification_report(y_test, y_pred_rf, target_names=['Control', 'Test']))

## 5. ROC Curve Comparison


In [ ]:
plt.figure(figsize=(8, 6))

models = [
    ('Logistic Regression', y_prob_lr, auc_lr),
    ('Decision Tree', y_prob_dt, auc_dt),
    ('Random Forest', y_prob_rf, auc_rf)
]

for name, probs, auc in models:
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Model Comparison Summary


In [ ]:
# Cross-validation scores
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_lr = cross_val_score(LogisticRegression(max_iter=1000), X_train_scaled, y_train, cv=cv, scoring='roc_auc')
cv_dt = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=42), X_train, y_train, cv=cv, scoring='roc_auc')
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), X_train, y_train, cv=cv, scoring='roc_auc')

comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Test Accuracy': [acc_lr, acc_dt, acc_rf],
    'Test AUC': [auc_lr, auc_dt, auc_rf],
    'CV AUC Mean': [cv_lr.mean(), cv_dt.mean(), cv_rf.mean()],
    'CV AUC Std': [cv_lr.std(), cv_dt.std(), cv_rf.std()]
})

comparison = comparison.round(4)
print(comparison.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 5))
x = np.arange(len(comparison))
width = 0.35

plt.bar(x - width/2, comparison['Test Accuracy'], width, label='Accuracy', color='steelblue')
plt.bar(x + width/2, comparison['Test AUC'], width, label='ROC-AUC', color='salmon')

plt.xticks(x, comparison['Model'])
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.ylim(0, 1.1)
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)


In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances_sorted.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('Random Forest - Feature Importance')
plt.tight_layout()
plt.show()

print('Feature Importances:')
print(importances.sort_values(ascending=False).round(4))

In [ ]:
# Logistic Regression Coefficients
lr_coef = pd.Series(
    np.abs(log_reg.coef_[0]),
    index=feature_cols
).sort_values(ascending=False)

print('Logistic Regression - Absolute Coefficients (after scaling):')
print(lr_coef.round(4))

## INTERPRETATION

**Classification Goal:** We trained models to distinguish whether a campaign day is from the Control or Test group based solely on performance metrics.

**Key Finding:** If models achieve high accuracy (significantly above 50%), it means the two campaign groups produce measurably different patterns in marketing metrics — implying the campaigns truly differ in their marketing funnel behavior.

**Feature Importance:** Features with high importance (e.g., `reach`, `impressions`, `spend_usd`) are the most differentiating metrics between the two campaign types.

**Model Performance:** Random Forest typically achieves the best performance due to its ensemble nature. However, with small sample sizes (~60 rows), results may vary significantly between train/test splits.

**Practical Use:** These models could be deployed to automatically classify new campaign days into control/test groups, or to identify which metrics best characterize each campaign type for future campaign planning.
